# Cryptocurrency Pairs Discovery System
## Implementation of Cornell CFEM Funnel (Section 4)

**This notebook implements the complete selection funnel:**

1. **Filter 1:** Sufficient historical depth (>1000 days)
2. **Filter 2:** Same blockchain/ecosystem
3. **Filter 3:** High correlation (log price > 0.9, returns > 0.5)
4. **Filter 4:** Z-Score Information Coefficient test
5. **Cointegration analysis**
6. **Rank by mean-reversion strength**
7. **Backtest top N pairs**

**Starting universe:** All Binance perpetual futures pairs  
**Output:** Top 5-10 tradable pairs with backtest results

In [16]:
import ccxt
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from statsmodels.tsa.stattools import adfuller
from statsmodels.regression.linear_model import OLS
from itertools import combinations
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-darkgrid')
print("✓ Libraries loaded")

✓ Libraries loaded


## Step 1: Collect Universe of Tokens

In [17]:
class BinanceFuturesUniverse:
    """Collect and filter Binance perpetual futures universe"""
    
    def __init__(self):
        self.exchange = ccxt.binance({
            'enableRateLimit': True,
            'options': {'defaultType': 'future'}
        })
    
    def get_all_usdt_perps(self):
        """Get all USDT-margined perpetual futures"""
        markets = self.exchange.load_markets()
        
        usdt_perps = []
        for symbol, market in markets.items():
            if (market['type'] == 'swap' and 
                market['quote'] == 'USDT' and 
                market['settle'] == 'USDT' and
                market['active']):
                usdt_perps.append({
                    'symbol': symbol,
                    'base': market['base'],
                    'quote': market['quote']
                })
        
        print(f"Found {len(usdt_perps)} USDT perpetual futures")
        return pd.DataFrame(usdt_perps)
    
    def fetch_ohlcv(self, symbol, days_back=400, limit=1500):
        """Fetch historical data for a symbol"""
        try:
            all_data = []
            end_date = datetime.now()
            start_date = end_date - timedelta(days=days_back)
            since = int(start_date.timestamp() * 1000)
            
            while True:
                ohlcv = self.exchange.fetch_ohlcv(symbol, '1h', since, limit)
                if not ohlcv or len(ohlcv) == 0:
                    break
                
                df = pd.DataFrame(ohlcv, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
                df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
                df.set_index('timestamp', inplace=True)
                all_data.append(df)
                
                since = int(df.index[-1].timestamp() * 1000) + 3600000
                if df.index[-1] >= end_date:
                    break
            
            if all_data:
                combined = pd.concat(all_data)
                combined = combined[~combined.index.duplicated(keep='first')]
                return combined
        except Exception as e:
            return None
        
        return None

# Initialize
universe = BinanceFuturesUniverse()
all_symbols = universe.get_all_usdt_perps()

print(f"\nSample symbols:")
print(all_symbols.head(10))

Found 560 USDT perpetual futures

Sample symbols:
           symbol  base quote
0   BTC/USDT:USDT   BTC  USDT
1   ETH/USDT:USDT   ETH  USDT
2   BCH/USDT:USDT   BCH  USDT
3   XRP/USDT:USDT   XRP  USDT
4   LTC/USDT:USDT   LTC  USDT
5   TRX/USDT:USDT   TRX  USDT
6   ETC/USDT:USDT   ETC  USDT
7  LINK/USDT:USDT  LINK  USDT
8   XLM/USDT:USDT   XLM  USDT
9   ADA/USDT:USDT   ADA  USDT


## Step 2: Filter 1 - Sufficient Historical Depth

**Paper requirement:** >1000 days of data  
**Our requirement:** >300 days (adjust based on what's available)

In [18]:
# Select top tokens to test (adjust this list based on your needs)
# Start with popular tokens that likely have long history
CANDIDATE_TOKENS = [
    'BTC', 'ETH', 'BNB', 'SOL', 'ADA', 'AVAX', 'DOT', 'MATIC', 'LINK',
    'UNI', 'ATOM', 'XRP', 'DOGE', 'SHIB', 'LTC', 'BCH', 'ETC', 'FIL',
    'AAVE', 'ALGO', 'SAND', 'MANA', 'AXS', 'GALA', 'APE', 'CRV', 'LDO',
    'OP', 'ARB', 'IMX', 'INJ', 'APT', 'SUI', 'SEI', 'TIA', 'ORDI',
    'BAND', 'NKN', 'ANKR', 'ONT', 'CTSI', 'CHR', 'ZIL', 'ATA', 'STORJ',
    'BAT', 'ENJ', 'SKL', 'ROSE', 'CELR', 'ICP', 'NEAR', 'FTM', 'ONE',
    'RUNE', 'KAVA', 'ZRX', 'GRT', 'SNX', 'COMP', 'YFI', 'SUSHI', 'BAL'
]

print(f"Testing {len(CANDIDATE_TOKENS)} candidate tokens for historical depth...\n")

valid_tokens = []
token_data = {}

for token in tqdm(CANDIDATE_TOKENS, desc="Fetching data"):
    symbol = f"{token}/USDT:USDT"
    
    # Check if symbol exists
    if symbol not in all_symbols['symbol'].values:
        continue
    
    data = universe.fetch_ohlcv(symbol, days_back=400)
    
    if data is not None and len(data) >= 7200:  # ~300 days of hourly data
        valid_tokens.append(token)
        token_data[token] = data['close']
        print(f"  ✓ {token}: {len(data)} hours ({len(data)/24:.0f} days)")

print(f"\n✓ Filter 1 complete: {len(valid_tokens)} tokens with sufficient history")
print(f"  Can create {len(valid_tokens) * (len(valid_tokens) - 1) // 2} pairs")

Fetching data:   0%|          | 0/63 [00:00<?, ?it/s]

Testing 63 candidate tokens for historical depth...



Fetching data:   2%|▏         | 1/63 [00:02<02:44,  2.65s/it]

  ✓ BTC: 9600 hours (400 days)


Fetching data:   3%|▎         | 2/63 [00:05<02:55,  2.88s/it]

...

  ✓ COMP: 9600 hours (400 days)


Fetching data:  97%|█████████▋| 61/63 [02:41<00:05,  2.76s/it]

  ✓ YFI: 9600 hours (400 days)


Fetching data: 100%|██████████| 63/63 [02:44<00:00,  2.61s/it]

  ✓ SUSHI: 9600 hours (400 days)

✓ Filter 1 complete: 58 tokens with sufficient history
  Can create 1653 pairs


## Step 3: Filter 3 - Correlation Screening

**Paper thresholds:**
- Log price correlation > 0.9
- Return correlation > 0.5

In [19]:
class CorrelationFilter:
    """Filter pairs by correlation (Paper Section 4.5)"""
    
    @staticmethod
    def test_correlation(price_a, price_b, 
                        logp_threshold=0.9, 
                        ret_threshold=0.5):
        """Test if pair meets correlation requirements"""
        
        # Align data
        merged = pd.DataFrame({'A': price_a, 'B': price_b}).dropna()
        
        if len(merged) < 100:
            return None
        
        # Log price correlation
        log_a = np.log(merged['A'])
        log_b = np.log(merged['B'])
        corr_logp = log_a.corr(log_b)
        
        # Return correlation
        ret_a = merged['A'].pct_change()
        ret_b = merged['B'].pct_change()
        corr_ret = ret_a.corr(ret_b)
        
        # Check thresholds
        passes = (corr_logp > logp_threshold and corr_ret > ret_threshold)
        
        return {
            'corr_logp': corr_logp,
            'corr_ret': corr_ret,
            'passes': passes,
            'n_obs': len(merged)
        }

# Test all pairs
print("Testing correlation for all pairs...\n")

correlation_results = []
total_pairs = len(list(combinations(valid_tokens, 2)))

for token_a, token_b in tqdm(combinations(valid_tokens, 2), 
                            total=total_pairs,
                            desc="Testing correlations"):
    
    result = CorrelationFilter.test_correlation(
        token_data[token_a], 
        token_data[token_b]
    )
    
    if result and result['passes']:
        correlation_results.append({
            'pair': f"{token_a}-{token_b}",
            'token_a': token_a,
            'token_b': token_b,
            'corr_logp': result['corr_logp'],
            'corr_ret': result['corr_ret'],
            'n_obs': result['n_obs']
        })

corr_df = pd.DataFrame(correlation_results)

if len(corr_df) > 0:
    corr_df = corr_df.sort_values('corr_logp', ascending=False)
    
    print(f"\n✓ Filter 3 complete: {len(corr_df)} pairs pass correlation filters")
    print(f"\nTop 10 by log-price correlation:")
    print(corr_df.head(10).to_string(index=False))
else:
    print("\n⚠️ No pairs passed correlation filters! Try lowering thresholds.")

Testing correlations:   2%|▏         | 37/1653 [00:00<00:04, 367.20it/s]

Testing correlation for all pairs...



Testing correlations: 100%|██████████| 1653/1653 [00:03<00:00, 479.52it/s]


✓ Filter 3 complete: 855 pairs pass correlation filters

Top 10 by log-price correlation:
      pair token_a token_b  corr_logp  corr_ret  n_obs
ANKR-STORJ    ANKR   STORJ   0.994253  0.667259   9600
  ANKR-ATA    ANKR     ATA   0.994110  0.619651   9600
  ANKR-ONE    ANKR     ONE   0.993193  0.705302   9600
  GALA-GRT    GALA     GRT   0.993190  0.893119   9600
 SAND-GALA    SAND    GALA   0.992982  0.897585   9600
 ATA-STORJ     ATA   STORJ   0.992869  0.615764   9600
 ONT-STORJ     ONT   STORJ   0.992769  0.660658   9600
   ZIL-ONE     ZIL     ONE   0.992446  0.733153   9600
 STORJ-ONE   STORJ     ONE   0.992214  0.697680   9600
   ATA-ONE     ATA     ONE   0.992114  0.712186   9600


## Step 4: Cointegration Testing

**Engle-Granger two-step procedure**

In [20]:
class CointegrationTester:
    """Test for cointegration (Paper Section 3.1)"""
    
    @staticmethod
    def test_cointegration(price_a, price_b):
        """Engle-Granger cointegration test"""
        
        # Align
        merged = pd.DataFrame({'A': price_a, 'B': price_b}).dropna()
        
        if len(merged) < 100:
            return None
        
        # Step 1: OLS regression
        log_a = np.log(merged['A'])
        log_b = np.log(merged['B'])
        
        X = np.column_stack([np.ones(len(log_b)), log_b])
        model = OLS(log_a, X)
        results = model.fit()
        
        alpha = results.params[0]
        beta = results.params[1]
        residuals = results.resid
        
        # Step 2: ADF test on residuals
        try:
            adf_result = adfuller(residuals, maxlag=1)
            adf_stat = adf_result[0]
            p_value = adf_result[1]
            is_cointegrated = p_value < 0.05
        except:
            return None
        
        return {
            'beta': beta,
            'alpha': alpha,
            'residual_std': residuals.std(),
            'adf_statistic': adf_stat,
            'adf_pvalue': p_value,
            'is_cointegrated': is_cointegrated
        }

# Test cointegration for correlation-filtered pairs
print("Testing cointegration...\n")

coint_results = []

for idx, row in tqdm(corr_df.iterrows(), total=len(corr_df), desc="Cointegration tests"):
    token_a = row['token_a']
    token_b = row['token_b']
    
    coint = CointegrationTester.test_cointegration(
        token_data[token_a],
        token_data[token_b]
    )
    
    if coint:
        coint_results.append({
            'pair': row['pair'],
            'token_a': token_a,
            'token_b': token_b,
            'corr_logp': row['corr_logp'],
            'corr_ret': row['corr_ret'],
            'beta': coint['beta'],
            'residual_std': coint['residual_std'],
            'adf_stat': coint['adf_statistic'],
            'adf_pvalue': coint['adf_pvalue'],
            'cointegrated': coint['is_cointegrated']
        })

coint_df = pd.DataFrame(coint_results)

if len(coint_df) > 0:
    # Sort by ADF statistic (more negative = stronger cointegration)
    coint_df = coint_df.sort_values('adf_stat', ascending=True)
    
    n_coint = coint_df['cointegrated'].sum()
    
    print(f"\n✓ Cointegration testing complete")
    print(f"  Formally cointegrated (p<0.05): {n_coint}/{len(coint_df)}")
    print(f"\nTop 10 by ADF statistic (most negative = best):")
    print(coint_df[['pair', 'adf_stat', 'adf_pvalue', 'cointegrated', 'corr_logp']].head(10).to_string(index=False))
else:
    print("\n⚠️ No pairs passed cointegration test")

Cointegration tests:   1%|          | 5/855 [00:00<00:18, 45.67it/s]

Testing cointegration...



Cointegration tests: 100%|██████████| 855/855 [00:16<00:00, 51.42it/s]



✓ Cointegration testing complete
  Formally cointegrated (p<0.05): 471/855

Top 10 by ADF statistic (most negative = best):
      pair  adf_stat   adf_pvalue  cointegrated  corr_logp
ANKR-STORJ -7.967736 2.835065e-12          True   0.994253
 ONT-STORJ -7.364716 9.301286e-11          True   0.992769
 ATA-STORJ -7.097165 4.259726e-10          True   0.992869
 STORJ-ENJ -6.677062 4.437714e-09          True   0.990645
 STORJ-ONE -6.663008 4.794316e-09          True   0.992214
  ANKR-ENJ -6.370596 2.350353e-08          True   0.991404
 ZIL-STORJ -6.138783 8.065450e-08          True   0.988284
   ONT-ATA -6.005981 1.615167e-07          True   0.991888
  ANKR-ATA -5.920035 2.519156e-07          True   0.994110
  ANKR-ONT -5.907108 2.692403e-07          True   0.991365


## Step 5: Information Coefficient (IC) Test

**Paper's Filter 4 (Section 4.6):** Test if Z-score predicts future spread returns

In [21]:
class ICTester:
    """Information Coefficient test (Paper Section 4.6)"""
    
    @staticmethod
    def compute_ic(price_a, price_b, 
                   window=4320,  # 6 months
                   forecast_horizon=24):  # 1 day ahead
        """Compute IC: correlation between Z-score and future spread return"""
        
        merged = pd.DataFrame({'A': price_a, 'B': price_b}).dropna()
        
        if len(merged) < window + forecast_horizon + 100:
            return None
        
        log_a = np.log(merged['A'])
        log_b = np.log(merged['B'])
        
        z_scores = []
        future_returns = []
        
        for i in range(window, len(merged) - forecast_horizon):
            # Rolling regression
            y = log_a.iloc[i-window:i]
            x = log_b.iloc[i-window:i]
            
            X = np.column_stack([np.ones(len(x)), x])
            model = OLS(y, X)
            fit = model.fit()
            
            alpha, beta = fit.params[0], fit.params[1]
            
            # Current Z-score
            resid = log_a.iloc[i] - alpha - beta * log_b.iloc[i]
            z = resid / fit.resid.std() if fit.resid.std() > 0 else 0
            
            # Future spread return
            future_log_a = log_a.iloc[i + forecast_horizon]
            future_log_b = log_b.iloc[i + forecast_horizon]
            
            ret_a = future_log_a - log_a.iloc[i]
            ret_b = future_log_b - log_b.iloc[i]
            spread_ret = ret_a - beta * ret_b
            
            z_scores.append(z)
            future_returns.append(spread_ret)
        
        # IC = correlation between Z and future spread return
        # Negative IC means mean reversion (high Z → negative future return)
        ic = np.corrcoef(z_scores, future_returns)[0, 1]
        
        return {
            'ic': ic,
            'n_samples': len(z_scores)
        }

# Test IC for top cointegrated pairs (this is slow!)
print("Computing Information Coefficient (this may take a while)...\n")

# Test top 20 pairs only to save time
top_pairs = coint_df.head(20)

ic_results = []

for idx, row in tqdm(top_pairs.iterrows(), total=len(top_pairs), desc="IC tests"):
    token_a = row['token_a']
    token_b = row['token_b']
    
    ic_result = ICTester.compute_ic(
        token_data[token_a],
        token_data[token_b],
        window=4320,
        forecast_horizon=24
    )
    
    if ic_result:
        ic_results.append({
            'pair': row['pair'],
            'token_a': token_a,
            'token_b': token_b,
            'ic': ic_result['ic'],
            'adf_stat': row['adf_stat'],
            'cointegrated': row['cointegrated'],
            'corr_logp': row['corr_logp']
        })

ic_df = pd.DataFrame(ic_results)

if len(ic_df) > 0:
    # Sort by IC (most negative = best mean reversion)
    ic_df = ic_df.sort_values('ic', ascending=True)
    
    print(f"\n✓ IC testing complete")
    print(f"\nTop 10 pairs by IC (most negative = strongest mean reversion):")
    print(ic_df[['pair', 'ic', 'adf_stat', 'cointegrated', 'corr_logp']].head(10).to_string(index=False))
    
    # Save results
    ic_df.to_csv('discovered_pairs.csv', index=False)
    print("\n✓ Results saved to discovered_pairs.csv")
else:
    print("\n⚠️ IC test failed for all pairs")

IC tests:   0%|          | 0/20 [00:00<?, ?it/s]

Computing Information Coefficient (this may take a while)...



IC tests: 100%|██████████| 20/20 [03:41<00:00, 11.08s/it]


✓ IC testing complete

Top 10 pairs by IC (most negative = strongest mean reversion):
      pair        ic  adf_stat  cointegrated  corr_logp
 ATA-STORJ -0.325037 -7.097165          True   0.992869
ANKR-STORJ -0.321200 -7.967736          True   0.994253
 STORJ-ENJ -0.307627 -6.677062          True   0.990645
   ONT-ATA -0.301986 -6.005981          True   0.991888
   DOT-ZRX -0.270999 -5.824690          True   0.988734
   ONT-ONE -0.270752 -5.593111          True   0.991662
 ONT-STORJ -0.267536 -7.364716          True   0.992769
CTSI-SUSHI -0.262298 -5.488098          True   0.975941
   ONT-ZIL -0.258852 -5.632756          True   0.988838
  ANKR-ENJ -0.258155 -6.370596          True   0.991404

✓ Results saved to discovered_pairs.csv


## Step 6: Backtest Top Pairs

In [22]:
class QuickBacktester:
    """Quick backtest for pair comparison"""
    
    @staticmethod
    def backtest(price_a, price_b,
                initial_capital=10000,
                leverage=3,
                window=4320,
                z_entry=1.5,  # Higher to reduce overtrading
                z_exit=0.0,
                stop_loss_pct=0.15,
                fee_rate=0.00045):
        
        merged = pd.DataFrame({'A': price_a, 'B': price_b}).dropna()
        
        if len(merged) < window + 100:
            return None
        
        log_a = np.log(merged['A'])
        log_b = np.log(merged['B'])
        
        equity = initial_capital
        position = 0
        trades = []
        equity_curve = []
        
        for i in range(window, len(merged)):
            # Rolling regression
            y = log_a.iloc[i-window:i]
            x = log_b.iloc[i-window:i]
            X = np.column_stack([np.ones(len(x)), x])
            model = OLS(y, X)
            fit = model.fit()
            alpha, beta = fit.params[0], fit.params[1]
            
            # Z-score
            resid = log_a.iloc[i] - alpha - beta * log_b.iloc[i]
            z = resid / fit.resid.std() if fit.resid.std() > 0 else 0
            
            current_price_a = merged['A'].iloc[i]
            current_price_b = merged['B'].iloc[i]
            
            # Mark-to-market
            if position != 0:
                ret_a = np.log(current_price_a / entry_price_a)
                ret_b = np.log(current_price_b / entry_price_b)
                spread_ret = ret_a - entry_beta * ret_b
                pnl = position * (entry_equity * leverage) * spread_ret / (1 + entry_beta)
                equity = entry_equity + pnl
                
                # Stop loss
                if (equity - entry_equity) / entry_equity < -stop_loss_pct:
                    equity -= 2 * 2 * fee_rate * entry_equity * leverage
                    equity = max(equity, initial_capital * 0.10)
                    trades.append({'pnl': equity - entry_equity, 'reason': 'STOP'})
                    position = 0
            
            # Entry
            if position == 0 and abs(z) > z_entry:
                entry_price_a = current_price_a
                entry_price_b = current_price_b
                entry_beta = beta
                entry_equity = equity
                equity -= 2 * 2 * fee_rate * equity * leverage
                position = -1 if z > z_entry else 1
            
            # Exit
            elif position != 0:
                if (position == 1 and z >= -z_exit) or (position == -1 and z <= z_exit):
                    equity -= 2 * 2 * fee_rate * entry_equity * leverage
                    trades.append({'pnl': equity - entry_equity, 'reason': 'NORMAL'})
                    position = 0
            
            equity_curve.append(equity)
        
        # Metrics
        equity_series = pd.Series(equity_curve)
        returns = equity_series.pct_change().dropna()
        
        total_return = (equity / initial_capital - 1) * 100
        sharpe = returns.mean() / returns.std() * np.sqrt(24 * 365) if returns.std() > 0 else 0
        
        cum_max = equity_series.cummax()
        dd = (equity_series - cum_max) / cum_max * 100
        max_dd = dd.min()
        
        return {
            'total_return': total_return,
            'sharpe': sharpe,
            'max_dd': max_dd,
            'num_trades': len(trades),
            'final_equity': equity
        }

# Backtest top 5 pairs
print("\nBacktesting top 5 pairs...\n")

backtest_results = []

for idx, row in ic_df.head(5).iterrows():
    print(f"Testing {row['pair']}...")
    
    result = QuickBacktester.backtest(
        token_data[row['token_a']],
        token_data[row['token_b']],
        leverage=3,
        z_entry=1.5,
        stop_loss_pct=0.15
    )
    
    if result:
        backtest_results.append({
            'pair': row['pair'],
            'ic': row['ic'],
            'sharpe': result['sharpe'],
            'return_pct': result['total_return'],
            'max_dd_pct': result['max_dd'],
            'trades': result['num_trades'],
            'final': result['final_equity']
        })
        
        print(f"  Sharpe: {result['sharpe']:.2f} | Return: {result['total_return']:.1f}% | DD: {result['max_dd']:.1f}%")

bt_df = pd.DataFrame(backtest_results)

if len(bt_df) > 0:
    bt_df = bt_df.sort_values('sharpe', ascending=False)
    
    print("\n" + "="*80)
    print("FINAL RANKING - TOP PAIRS")
    print("="*80)
    print(bt_df.to_string(index=False))
    print("="*80)
    
    bt_df.to_csv('backtest_results.csv', index=False)
    print("\n✓ Backtest results saved")


Backtesting top 5 pairs...

Testing ATA-STORJ...
  Sharpe: 2.46 | Return: 225.2% | DD: -34.9%
Testing ANKR-STORJ...
  Sharpe: 2.64 | Return: 269.0% | DD: -28.0%
Testing STORJ-ENJ...
  Sharpe: 3.79 | Return: 628.9% | DD: -29.8%
Testing ONT-ATA...
  Sharpe: 2.70 | Return: 290.7% | DD: -55.0%
Testing DOT-ZRX...
  Sharpe: 3.02 | Return: 258.9% | DD: -37.2%

FINAL RANKING - TOP PAIRS
      pair        ic   sharpe  return_pct  max_dd_pct  trades        final
 STORJ-ENJ -0.307627 3.790835  628.856785  -29.789978      21 72885.678517
   DOT-ZRX -0.270999 3.022702  258.946920  -37.218287      17 35894.691984
   ONT-ATA -0.301986 2.704462  290.718023  -55.011885      17 39071.802284
ANKR-STORJ -0.321200 2.644454  268.957092  -28.022691      17 36895.709215
 ATA-STORJ -0.325037 2.460259  225.166207  -34.852645      17 32516.620735

✓ Backtest results saved


## Step 7: 70:30 In/Out-Sample Backtest + Fee Sensitivity

- Uses the same `QuickBacktester` logic as Step 6
- Splits each pair into 70% in-sample and 30% out-sample
- Evaluates fee sensitivity at **4.5 bps (baseline), 5 bps, and 10 bps**
- `window_days` is explicit and saved in output
- Default `window_days = 60` (hourly bars); edit this value as needed


In [24]:
# Step 7: Same backtest as Step 6 with 70:30 in/out split + fee scenarios
print("\nStep 7: 70:30 in/out-sample backtest with fee sensitivity")

split_ratio = 0.7
# Explicit rolling-window length in days (hourly bars)
window_days = 90
window = int(window_days * 24)

# Baseline from Step 6 is 4.5 bps; add 5 bps and 10 bps
fee_bps_list = [4.5, 5.0, 10.0]

print(f"Window length: {window} hours ({window_days:.1f} days)")
print(f"In-sample / Out-sample: {int(split_ratio*100)}:{int((1-split_ratio)*100)}")
print("Fee scenarios (bps): " + ", ".join([f"{x:g}" for x in fee_bps_list]))

split_rows = []
min_points = window + 100
min_days_per_split = min_points / 24

for _, row in ic_df.head(5).iterrows():
    pair = row["pair"]
    token_a = row["token_a"]
    token_b = row["token_b"]

    merged = pd.DataFrame({"A": token_data[token_a], "B": token_data[token_b]}).dropna()
    split_idx = int(len(merged) * split_ratio)
    in_df = merged.iloc[:split_idx].copy()
    out_df = merged.iloc[split_idx:].copy()

    in_days = len(in_df) / 24
    out_days = len(out_df) / 24
    print(f"\nTesting {pair} | in={in_days:.1f}d, out={out_days:.1f}d")

    if len(in_df) < min_points or len(out_df) < min_points:
        print(f"  Skipped: insufficient split length for {window_days:.1f}d window (need >= {min_days_per_split:.1f}d each split)")
        continue

    for sample_name, sample_df in [("in_sample", in_df), ("out_sample", out_df)]:
        for fee_bps in fee_bps_list:
            result = QuickBacktester.backtest(
                sample_df["A"],
                sample_df["B"],
                leverage=3,
                window=window,
                z_entry=1.5,
                stop_loss_pct=0.15,
                fee_rate=fee_bps / 10000
            )

            if result is None:
                continue

            split_rows.append({
                "pair": pair,
                "ic": row["ic"],
                "sample": sample_name,
                "fee_bps": fee_bps,
                "window_days": window_days,
                "sample_days": len(sample_df) / 24,
                "sharpe": result["sharpe"],
                "return_pct": result["total_return"],
                "max_dd_pct": result["max_dd"],
                "trades": result["num_trades"],
                "final": result["final_equity"]
            })

split_bt_df = pd.DataFrame(split_rows)

if len(split_bt_df) > 0:
    split_bt_df = split_bt_df.sort_values(["sample", "fee_bps", "sharpe"], ascending=[True, True, False])
    print("\n" + "="*100)
    print("STEP 7 RESULTS - 70:30 IN/OUT SAMPLE + FEE SENSITIVITY")
    print("="*100)
    print(split_bt_df[["pair", "sample", "fee_bps", "window_days", "sample_days", "sharpe", "return_pct", "max_dd_pct", "trades"]].to_string(index=False))
    print("="*100)

    split_bt_df.to_csv("backtest_results_70_30_fee_sensitivity.csv", index=False)
    print("\n✓ Step 7 results saved to backtest_results_70_30_fee_sensitivity.csv")
else:
    print("\n⚠️ No Step 7 backtest results were generated. Reduce window_days or increase historical data.")



Step 7: 70:30 in/out-sample backtest with fee sensitivity
Window length: 2160 hours (90.0 days)
In-sample / Out-sample: 70:30
Fee scenarios (bps): 4.5, 5, 10

Testing ATA-STORJ | in=280.0d, out=120.0d

Testing ANKR-STORJ | in=280.0d, out=120.0d

Testing STORJ-ENJ | in=280.0d, out=120.0d

Testing ONT-ATA | in=280.0d, out=120.0d

Testing DOT-ZRX | in=280.0d, out=120.0d

STEP 7 RESULTS - 70:30 IN/OUT SAMPLE + FEE SENSITIVITY
      pair     sample  fee_bps  window_days  sample_days    sharpe  return_pct  max_dd_pct  trades
ANKR-STORJ  in_sample      4.5           90        280.0  1.659466   61.393641  -28.456515      16
 STORJ-ENJ  in_sample      4.5           90        280.0  1.624226   55.743397  -29.187642      13
   DOT-ZRX  in_sample      4.5           90        280.0  1.520681   47.691823  -31.039485      10
   ONT-ATA  in_sample      4.5           90        280.0  1.256856   43.231378  -53.010552       9
 ATA-STORJ  in_sample      4.5           90        280.0  1.106830   34.180712

## Summary & Recommendations

### **What this notebook does:**

1. ✅ Fetches all Binance perpetual futures
2. ✅ Filters for sufficient history (300+ days)
3. ✅ Tests correlation (log price > 0.9, returns > 0.5)
4. ✅ Tests cointegration (Engle-Granger ADF)
5. ✅ Computes Information Coefficient (Z-score predictiveness)
6. ✅ Backtests top pairs with conservative settings
7. ✅ Runs 70:30 in/out-sample backtest with 4.5/5/10 bps fee sensitivity

### **Parameters used:**
- 3x leverage (vs 5x in original)
- 1.5 Z-entry (vs 1.0 - reduces overtrading)
- 15% stop loss
- Step 6 rolling window: 6 months
- Step 7 rolling window: configurable `window_days` (default 60 days)

### **How to use results:**

Look for pairs with:
- ✅ **IC < -0.15** (strong mean reversion)
- ✅ **Sharpe > 0.5** in backtest
- ✅ **Max DD < -40%**
- ✅ **10-50 trades** (not too many, not too few)

### **Next steps:**
1. Review `discovered_pairs.csv` and `backtest_results.csv`
2. Review `backtest_results_70_30_fee_sensitivity.csv` for in/out-sample and fee impact
3. Pick top 2-3 pairs
4. Run detailed backtest with parameter grid search
5. Paper trade for 1-2 weeks
6. Go live with small capital

**This approach finds pairs that ACTUALLY work in YOUR data period, not just what worked in the paper's 2023-2025 data!**
